## 准备数据

In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

In [2]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [3]:
class myModel:
    def __init__(self):
        ####################
        '''声明模型对应的参数'''
        # 第一层的权重和偏置
        self.W1 = tf.Variable(tf.random.normal([28 * 28, 256], stddev=0.1))
        self.b1 = tf.Variable(tf.zeros([256]))
        
        # 第二层（输出层）的权重和偏置
        self.W2 = tf.Variable(tf.random.normal([256, 10], stddev=0.1))
        self.b2 = tf.Variable(tf.zeros([10]))
        ####################
    def __call__(self, x):
        ####################
        '''实现模型函数体，返回未归一化的logits'''
        # 展平输入图像：从 (Batch_size, 28, 28) 变成 (Batch_size, 784)
        x = tf.reshape(x, [-1, 28 * 28])
        
        # 第一层前向传播：线性计算 + ReLU激活
        h1 = tf.matmul(x, self.W1) + self.b1
        h1_relu = tf.nn.relu(h1)
        
        # 第二层前向传播：只做线性计算
        logits = tf.matmul(h1_relu, self.W2) + self.b2
        ####################
        return logits
        
model = myModel()

optimizer = optimizers.Adam()

## 计算 loss

In [4]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    # compute gradient
    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    for g, v in zip(grads, trainable_vars):
        v.assign_sub(0.01*g)

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [5]:
train_data, test_data = mnist_dataset()
for epoch in range(50):
    loss, accuracy = train_one_step(model, optimizer, 
                                    tf.constant(train_data[0], dtype=tf.float32), 
                                    tf.constant(train_data[1], dtype=tf.int64))
    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())
loss, accuracy = test(model, 
                      tf.constant(test_data[0], dtype=tf.float32), 
                      tf.constant(test_data[1], dtype=tf.int64))

print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

epoch 0 : loss 2.837712 ; accuracy 0.060983334
epoch 1 : loss 2.752419 ; accuracy 0.0591
epoch 2 : loss 2.6876915 ; accuracy 0.06266667
epoch 3 : loss 2.6357882 ; accuracy 0.06881667
epoch 4 : loss 2.5922928 ; accuracy 0.07571667
epoch 5 : loss 2.5545714 ; accuracy 0.082883336
epoch 6 : loss 2.5210016 ; accuracy 0.09026667
epoch 7 : loss 2.4905245 ; accuracy 0.09681667
epoch 8 : loss 2.4624197 ; accuracy 0.1037
epoch 9 : loss 2.4361844 ; accuracy 0.11141667
epoch 10 : loss 2.4114516 ; accuracy 0.11855
epoch 11 : loss 2.3879488 ; accuracy 0.12618333
epoch 12 : loss 2.3654718 ; accuracy 0.13278334
epoch 13 : loss 2.3438578 ; accuracy 0.14036667
epoch 14 : loss 2.3229833 ; accuracy 0.14736667
epoch 15 : loss 2.3027492 ; accuracy 0.15418333
epoch 16 : loss 2.2830765 ; accuracy 0.16205
epoch 17 : loss 2.2639 ; accuracy 0.16986667
epoch 18 : loss 2.2451687 ; accuracy 0.1779
epoch 19 : loss 2.2268395 ; accuracy 0.18546666
epoch 20 : loss 2.2088783 ; accuracy 0.1944
epoch 21 : loss 2.1912556 ;